In [ ]:
# =========================
# INSTALL DEPENDENCIES
# =========================
!pip install opencv-python-headless imutils matplotlib

# =========================
# IMPORTS
# =========================
import cv2
import numpy as np
import imutils
import matplotlib.pyplot as plt
from google.colab import files

# =========================
# UPLOAD IMAGE
# =========================
files.upload()   # Upload an image named: document.jpg

# =========================
# UTILITY FUNCTIONS
# =========================
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]   # top-left
    rect[2] = pts[np.argmax(s)]   # bottom-right
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # top-right
    rect[3] = pts[np.argmax(diff)]  # bottom-left
    return rect


def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped

# =========================
# READ IMAGE
# =========================
image = cv2.imread("document.jpg")
if image is None:
    raise ValueError("document.jpg not found. Upload the image properly.")

orig = image.copy()
image = imutils.resize(image, height=500)

# =========================
# PREPROCESSING
# =========================
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
edged = cv2.Canny(blur, 75, 200)

plt.figure(figsize=(5,5))
plt.imshow(edged, cmap="gray")
plt.title("Edge Detection")
plt.axis("off")

# =========================
# FIND DOCUMENT CONTOUR
# =========================
cnts = cv2.findContours(edged.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
cnts = imutils.grab_contours(cnts)
cnts = sorted(cnts, key=cv2.contourArea, reverse=True)

docCnt = None
for c in cnts:
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    if len(approx) == 4:
        docCnt = approx
        break

if docCnt is None:
    raise Exception("No document detected. Use a clearer image.")

# =========================
# PERSPECTIVE TRANSFORM
# =========================
ratio = orig.shape[0] / 500.0
warped = four_point_transform(orig, docCnt.reshape(4, 2) * ratio)

# =========================
# SCAN EFFECT
# =========================
warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
scanned = cv2.adaptiveThreshold(
    warped_gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    2
)

# =========================
# DISPLAY RESULT
# =========================
plt.figure(figsize=(6,8))
plt.imshow(scanned, cmap="gray")
plt.title("Scanned Document")
plt.axis("off")
